# Caso 3 - Clustering: segmentación de universidades (College Data)
Se excluye la columna `College`.

In [1]:
# Funciones auxiliares — Problema 3: Clustering (College Data)
# Son las mismas que usamos en los laboratorios de K-Means, DBSCAN y PCA.
# Ejecuta esta celda una vez; después sólo llama a las funciones.
#   Nuevo: evaluar_clustering ahora reporta DOS silhouettes:
#        'silhouette'           → como la calcula sklearn: el ruido (-1) cuenta como un grupo más.
#        'silhouette_sin_ruido' → como en clase: sólo los puntos que quedaron en un cluster.
#   Nuevo: tabla_metricas_k usa n_init='auto' (el valor por defecto de KMeans) en vez de n_init=10.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Número de clusters (sin contar el ruido), % de ruido y silhouette de un agrupamiento."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    # sklearn: todas las etiquetas, el ruido incluido como un grupo (necesita al menos 2 etiquetas)
    fila['silhouette'] = (float(round(silhouette_score(X, labels), 3))
                          if len(np.unique(labels)) >= 2 else np.nan)
    # Sin ruido: sólo los puntos asignados a un cluster (necesita al menos 2 clusters)
    fila['silhouette_sin_ruido'] = (float(round(silhouette_score(X[mask], labels[mask]), 3))
                                    if n_clusters >= 2 else np.nan)
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con la inercia (para el codo) y la silhouette."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init='auto', random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), 'silhouette': fila['silhouette']})
    return pd.DataFrame(filas)


def grilla_dbscan(X, eps_values, min_samples_values):
    """Ajusta DBSCAN para cada combinación de eps y min_samples.
    Regresa una tabla con el número de clusters, % de ruido y ambas silhouettes."""
    filas = []
    for eps in eps_values:
        for ms in min_samples_values:
            labels = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
            fila = evaluar_clustering(f'eps={eps}, ms={ms}', X, labels)
            filas.append({'eps': eps, 'min_samples': ms, **fila})
    return pd.DataFrame(filas).drop(columns='modelo')


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta_k(tabla):
    """Silhouette promedio para cada k (tabla de tabla_metricas_k). Más alto es mejor."""
    fig = px.line(
        tabla,
        x='k',
        y='silhouette',
        markers=True,
        labels={'k': 'Número de clusters (k)', 'silhouette': 'Silhouette promedio'},
        title='Silhouette según k (↑ mejor)'
    )
    fig.update_xaxes(dtick=1)
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples. El 'codo' sugiere un buen eps; eps (opcional) dibuja una línea de referencia."""
    from sklearn.neighbors import NearestNeighbors

    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


In [2]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

df = pd.read_csv('https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/College_Data.csv')
X = df.drop(columns='College')
X['Private'] = X['Private'].map({'Yes': 1, 'No': 0})
print(X.shape)

(777, 18)


In [3]:
tabla_k = tabla_metricas_k(X, range(2, 9))
print(tabla_k)
print('Silhouette K-Means K=2 (sin escalar): %.2f' % tabla_k.loc[tabla_k.k == 2, 'silhouette'].iloc[0])

   k       inercia  silhouette
0  2  4.926657e+10       0.537
1  3  3.236587e+10       0.476
2  4  2.724896e+10       0.423
3  5  2.076756e+10       0.400
4  6  1.776536e+10       0.308
5  7  1.640310e+10       0.308
6  8  1.436988e+10       0.307
Silhouette K-Means K=2 (sin escalar): 0.54


In [4]:
X_scaled = StandardScaler().fit_transform(X)
labels = DBSCAN(eps=2.0, min_samples=5).fit_predict(X_scaled)
evaluar_clustering('DBSCAN eps=2.0, ms=5', X_scaled, labels)

{'modelo': 'DBSCAN eps=2.0, ms=5',
 'n_clusters': 3,
 'pct_ruido': 37.2,
 'silhouette': 0.041,
 'silhouette_sin_ruido': 0.256}

In [5]:
X_pca = PCA(n_components=0.8, random_state=42).fit_transform(X_scaled)
print('Componentes:', X_pca.shape[1])
km3 = KMeans(n_clusters=3, random_state=42, n_init='auto').fit(X_pca)
evaluar_clustering('K-Means k=3 sobre PCA', X_pca, km3.labels_)

Componentes: 6


{'modelo': 'K-Means k=3 sobre PCA',
 'n_clusters': 3,
 'pct_ruido': 0.0,
 'silhouette': 0.292,
 'silhouette_sin_ruido': 0.292}

**Respuestas conceptuales:** DBSCAN maneja mejor formas no esféricas como elipses; un `eps` muy pequeño marca muchos puntos como ruido; se prueban varios K para estimar la estructura óptima; la ventaja de DBSCAN con densidades variables es que detecta ruido.